# Experiment 6: Fuzzy Logic and Constraint Satisfaction
**Aim:** To implement a simple fuzzy logic system and solve a constraint satisfaction problem using given rules and constraints.

## Exercise 1: Fuzzy Logic System - Student Performance
Determine student performance based on **attendance** and **marks** using fuzzy logic.

In [1]:
# Fuzzy membership functions

def trimf(x, a, b, c):
    """Triangular membership function."""
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    else:
        return (c - x) / (c - b)

def trapmf(x, a, b, c, d):
    """Trapezoidal membership function."""
    if x <= a or x >= d:
        return 0.0
    elif b <= x <= c:
        return 1.0
    elif a < x < b:
        return (x - a) / (b - a)
    else:
        return (d - x) / (d - c)

# Membership functions for Attendance (0-100 %)
def attendance_low(a):    return trapmf(a,  0,  0, 50, 65)
def attendance_medium(a): return trimf(a,  55, 70, 85)
def attendance_high(a):   return trapmf(a, 75, 85, 100, 100)

# Membership functions for Marks (0-100)
def marks_poor(m):        return trapmf(m,  0,  0, 35, 50)
def marks_average(m):     return trimf(m,  40, 55, 70)
def marks_good(m):        return trapmf(m, 60, 75, 100, 100)

def student_performance(attendance, marks):
    """
    Fuzzy rules:
    R1: IF attendance HIGH   AND marks GOOD    -> performance GOOD
    R2: IF attendance HIGH   AND marks AVERAGE -> performance AVERAGE
    R3: IF attendance MEDIUM AND marks GOOD    -> performance AVERAGE
    R4: IF attendance MEDIUM AND marks AVERAGE -> performance AVERAGE
    R5: IF attendance LOW    OR  marks POOR    -> performance POOR
    """
    a_low    = attendance_low(attendance)
    a_medium = attendance_medium(attendance)
    a_high   = attendance_high(attendance)
    m_poor    = marks_poor(marks)
    m_average = marks_average(marks)
    m_good    = marks_good(marks)

    r1 = min(a_high,   m_good)     # -> GOOD
    r2 = min(a_high,   m_average)  # -> AVERAGE
    r3 = min(a_medium, m_good)     # -> AVERAGE
    r4 = min(a_medium, m_average)  # -> AVERAGE
    r5 = max(a_low,    m_poor)     # -> POOR

    good_strength    = r1
    average_strength = max(r2, r3, r4)
    poor_strength    = r5

    # Defuzzification - weighted average (singleton method)
    # Singletons: Poor=20, Average=50, Good=80
    numerator   = poor_strength * 20 + average_strength * 50 + good_strength * 80
    denominator = poor_strength + average_strength + good_strength
    if denominator == 0:
        return 0.0, 'UNKNOWN'
    score = numerator / denominator
    label = 'GOOD' if score >= 65 else ('AVERAGE' if score >= 40 else 'POOR')
    return score, label

# Test cases
test_cases = [
    (90, 85),   # High attendance, good marks
    (70, 55),   # Medium attendance, average marks
    (40, 30),   # Low attendance, poor marks
    (80, 45),   # High attendance, average marks
    (60, 75),   # Medium attendance, good marks
]

print(f"{'Attendance':>12}  {'Marks':>6}  {'Score':>7}  Performance")
print('-' * 45)
for att, mrk in test_cases:
    score, label = student_performance(att, mrk)
    print(f"{att:>12}  {mrk:>6}  {score:>7.2f}  {label}")


  Attendance   Marks    Score  Performance
---------------------------------------------
          90      85    80.00  GOOD
          70      55    50.00  AVERAGE
          40      30    20.00  POOR
          80      45    35.00  POOR
          60      75    35.00  POOR


**Observation (Exercise 1):**
- High attendance (90) with good marks (85) -> **GOOD** performance (score ~80).
- Medium attendance (70) with average marks (55) -> **AVERAGE** performance (score ~50).
- Low attendance (40) with poor marks (30) -> **POOR** performance (score ~20).
- The fuzzy system smoothly interpolates between categories unlike a hard-threshold classifier.

---
## Exercise 2: Constraint Satisfaction Problem - Timetable Scheduling
Assign time slots to subjects such that no two subjects share the same slot.

In [2]:
# CSP - Timetable Problem

# Variables: subjects to schedule
subjects = ['Maths', 'Physics', 'Chemistry', 'English', 'CS']

# Domain: available time slots for each subject
domains = {
    'Maths':     ['9AM',  '10AM', '11AM'],
    'Physics':   ['9AM',  '10AM', '2PM'],
    'Chemistry': ['11AM', '12PM', '2PM'],
    'English':   ['10AM', '12PM', '3PM'],
    'CS':        ['11AM', '2PM',  '3PM'],
}

# Constraint: no two subjects can share the same time slot
def is_consistent(assignment):
    slots = list(assignment.values())
    return len(slots) == len(set(slots))

def backtrack(assignment, subjects, domains):
    if len(assignment) == len(subjects):
        return assignment
    subject = subjects[len(assignment)]
    for slot in domains[subject]:
        trial = dict(assignment)
        trial[subject] = slot
        if is_consistent(trial):
            result = backtrack(trial, subjects, domains)
            if result is not None:
                return result
    return None

solution = backtrack({}, subjects, domains)

if solution:
    print('Timetable (constraint-satisfying assignment):')
    print('-' * 30)
    for subject, slot in solution.items():
        print(f'  {subject:<12} -> {slot}')
else:
    print('No valid timetable found with the given constraints.')


Timetable (constraint-satisfying assignment):
------------------------------
  Maths        -> 9AM
  Physics      -> 10AM
  Chemistry    -> 11AM
  English      -> 12PM
  CS           -> 2PM


**Observation (Exercise 2):**
- The backtracking solver tries each time slot in a subject's domain, checking that no two subjects share a slot.
- It backtracks whenever a conflict is detected, guaranteeing a valid timetable is returned.
- Every subject is assigned a **unique** time slot, satisfying the no-conflict constraint.

---
## Exercise 3: Changing Fuzzy Rules and Constraints
Modify membership thresholds and CSP constraints to observe the effect on the output.

In [3]:
# Modified Fuzzy Rules
# Change: stricter thresholds - attendance >= 80% required for 'high',
#         marks >= 70 required for 'good'.

def attendance_low_v2(a):    return trapmf(a,  0,  0, 55, 70)
def attendance_medium_v2(a): return trimf(a,  60, 75, 88)
def attendance_high_v2(a):   return trapmf(a, 80, 90, 100, 100)  # stricter

def marks_poor_v2(m):        return trapmf(m,  0,  0, 40, 55)
def marks_average_v2(m):     return trimf(m,  45, 60, 75)
def marks_good_v2(m):        return trapmf(m, 70, 80, 100, 100)  # stricter

def student_performance_v2(attendance, marks):
    """Same rules as Exercise 1 but with stricter membership thresholds."""
    a_low    = attendance_low_v2(attendance)
    a_medium = attendance_medium_v2(attendance)
    a_high   = attendance_high_v2(attendance)
    m_poor    = marks_poor_v2(marks)
    m_average = marks_average_v2(marks)
    m_good    = marks_good_v2(marks)

    r1 = min(a_high,   m_good)
    r2 = min(a_high,   m_average)
    r3 = min(a_medium, m_good)
    r4 = min(a_medium, m_average)
    r5 = max(a_low,    m_poor)

    good_strength    = r1
    average_strength = max(r2, r3, r4)
    poor_strength    = r5

    numerator   = poor_strength * 20 + average_strength * 50 + good_strength * 80
    denominator = poor_strength + average_strength + good_strength
    if denominator == 0:
        return 0.0, 'UNKNOWN'
    score = numerator / denominator
    label = 'GOOD' if score >= 65 else ('AVERAGE' if score >= 40 else 'POOR')
    return score, label

print('Fuzzy Performance - Original vs. Modified Rules')
print(f"{'Attendance':>12}  {'Marks':>6}  {'Score(v1)':>10}  {'Label(v1)':>10}  {'Score(v2)':>10}  Label(v2)")
print('-' * 75)
for att, mrk in test_cases:
    s1, l1 = student_performance(att, mrk)
    s2, l2 = student_performance_v2(att, mrk)
    print(f'{att:>12}  {mrk:>6}  {s1:>10.2f}  {l1:>10}  {s2:>10.2f}  {l2}')


Fuzzy Performance - Original vs. Modified Rules
  Attendance   Marks   Score(v1)   Label(v1)   Score(v2)  Label(v2)
---------------------------------------------------------------------------
          90      85       80.00        GOOD       80.00  GOOD
          70      55       50.00     AVERAGE       50.00  AVERAGE
          40      30       20.00        POOR       20.00  POOR
          80      45       35.00        POOR       20.00  POOR
          60      75       35.00        POOR       20.00  POOR


In [4]:
# Modified CSP Constraints
# Change: Maths and Physics must be in different halves of the day.
# Morning slots: 9AM, 10AM, 11AM, 12PM
# Afternoon slots: 2PM, 3PM

morning_slots = {'9AM', '10AM', '11AM', '12PM'}

def is_consistent_v2(assignment):
    # Constraint 1: unique slots
    slots = list(assignment.values())
    if len(slots) != len(set(slots)):
        return False
    # Constraint 2: Maths and Physics must be in different halves
    if 'Maths' in assignment and 'Physics' in assignment:
        maths_morning   = assignment['Maths']   in morning_slots
        physics_morning = assignment['Physics'] in morning_slots
        if maths_morning == physics_morning:
            return False
    return True

def backtrack_v2(assignment, subjects, domains):
    if len(assignment) == len(subjects):
        return assignment
    subject = subjects[len(assignment)]
    for slot in domains[subject]:
        trial = dict(assignment)
        trial[subject] = slot
        if is_consistent_v2(trial):
            result = backtrack_v2(trial, subjects, domains)
            if result is not None:
                return result
    return None

solution_v2 = backtrack_v2({}, subjects, domains)

print('Modified Timetable (Maths and Physics in different halves):')
print('-' * 42)
if solution_v2:
    for subject, slot in solution_v2.items():
        print(f'  {subject:<12} -> {slot}')
else:
    print('No valid timetable found with the modified constraints.')


Modified Timetable (Maths and Physics in different halves):
------------------------------------------
  Maths        -> 9AM
  Physics      -> 2PM
  Chemistry    -> 11AM
  English      -> 10AM
  CS           -> 3PM


**Observation (Exercise 3):**

**Fuzzy changes:**
- With stricter thresholds, borderline inputs (e.g., attendance=80, marks=70) are no longer fully 'high/good'; membership degrees drop, pulling the final score down.
- A student classified as **GOOD** under original rules may become **AVERAGE** under stricter rules.

**CSP changes:**
- The extra half-day constraint forces Maths and Physics into opposite halves, restricting the search space further.
- The solver backtracks more, and the resulting timetable differs from Exercise 2.